# 03b — Join-key autopsy

Run this after a zero-match result from notebook 03. It isolates which key
component fails: identifiers, dates, or clock. Monday only — fast.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np

orig = pd.read_parquet(os.path.join(C.INTERIM, 'original.parquet'))
impr = pd.read_parquet(os.path.join(C.INTERIM, 'improved.parquet'))
om, im = orig[orig['day'] == 'monday'].copy(), impr[impr['day'] == 'monday'].copy()
print(f'monday: original {len(om):,}   improved {len(im):,}')

Mounted at /content/drive
monday: original 529,918   improved 371,624


In [2]:
# --- 1. Raw samples side by side -------------------------------------------
# Look at these with your eyes before any statistics.
COLS = ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp']
have_o = [c for c in COLS if c in om.columns]
have_i = [c for c in COLS if c in im.columns]
print('original columns present:', have_o)
print('improved columns present:', have_i)

print('\nORIGINAL (3 rows):')
display(om[have_o].head(3))
print('IMPROVED (3 rows):')
display(im[have_i].head(3))

print('\ndtypes:')
print(pd.DataFrame({'original': om[have_o].dtypes.astype(str),
                    'improved': im[have_i].dtypes.astype(str)}))

original columns present: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp']
improved columns present: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp']

ORIGINAL (3 rows):


,src_ip,src_port,dst_ip,dst_port,protocol,timestamp
703245,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58
703246,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58
703247,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58


IMPROVED (3 rows):


,src_ip,src_port,dst_ip,dst_port,protocol,timestamp
547557,8.6.0.1,0,8.0.6.4,0,0,2017-07-03 11:56:34.157427
547558,192.168.10.9,123,192.168.10.3,123,17,2017-07-03 11:56:55.428911
547559,192.168.10.12,5353,224.0.0.251,5353,17,2017-07-03 11:57:21.057686



dtypes:
          original improved
src_ip      object   object
src_port   float64    int32
dst_ip      object   object
dst_port   float64    int32
protocol   float64    int32
timestamp   object   object


In [3]:
# --- 2. Component-by-component overlap --------------------------------------
# Cast exactly as the join does, then measure value-set overlap per column.
# The component with ~zero overlap is the killer.

def cast(s, kind):
    if kind == 'ip':
        return s.astype(str).str.strip()
    return pd.to_numeric(s, errors='coerce').astype('Int64').astype(str)

KINDS = {'src_ip': 'ip', 'dst_ip': 'ip',
         'src_port': 'num', 'dst_port': 'num', 'protocol': 'num'}

for c, kind in KINDS.items():
    if c not in om.columns or c not in im.columns:
        print(f'{c:10s} MISSING on one side')
        continue
    a, b = set(cast(om[c], kind)), set(cast(im[c], kind))
    inter = a & b
    print(f'{c:10s} original {len(a):>8,} distinct | improved {len(b):>8,} distinct '
          f'| shared {len(inter):>8,} '
          f'| share of original {len(inter)/max(len(a),1):6.1%}')
    if len(inter) / max(len(a), 1) < 0.5:
        print(f'           sample original: {sorted(a)[:5]}')
        print(f'           sample improved: {sorted(b)[:5]}')

src_ip     original    8,239 distinct | improved       90 distinct | shared       79 | share of original   1.0%
           sample original: ['1.9.56.32', '101.102.235.200', '101.254.102.143', '103.203.138.103', '103.228.55.44']
           sample improved: ['104.193.83.57', '104.44.81.176', '104.97.95.20', '125.212.216.123', '125.212.233.246']
dst_ip     original    9,698 distinct | improved    9,698 distinct | shared    9,698 | share of original 100.0%
src_port   original   52,723 distinct | improved   52,635 distinct | shared   52,634 | share of original  99.8%
dst_port   original   30,238 distinct | improved      642 distinct | shared      642 | share of original   2.1%
           sample original: ['0', '10002', '10139', '10157', '1028']
           sample improved: ['0', '10139', '10157', '10424', '10613']
protocol   original        3 distinct | improved        4 distinct | shared        3 | share of original 100.0%


In [4]:
# --- 3. The clock, in isolation ---------------------------------------------
# Dates first, then minute-set alignment across a WIDE shift range.

def choose_parse(series, name):
    a = pd.to_datetime(series, errors='coerce', dayfirst=True)
    b = pd.to_datetime(series, errors='coerce', dayfirst=False)
    va, vb = (a.dt.month == 7).mean(), (b.dt.month == 7).mean()
    ts, mode = (a, 'dayfirst=True') if va >= vb else (b, 'dayfirst=False')
    print(f'{name:9s} {mode:15s} month==7 {max(va,vb):6.1%}  NaT {ts.isna().mean():.2%}')
    return ts

ts_o = choose_parse(om['timestamp'], 'original')
ts_i = choose_parse(im['timestamp'], 'improved')

# 12h repair on whichever side shows the signature
for name in ['o', 'i']:
    ts = ts_o if name == 'o' else ts_i
    h = ts.dt.hour
    if h.between(1, 7).mean() > 0.02 and h.between(13, 17).mean() < 0.005:
        ts = ts + pd.to_timedelta(h.between(1, 7).astype('int64') * 12, unit='h')
        if name == 'o':
            ts_o = ts
        else:
            ts_i = ts
        print(f'12h repair applied to {"original" if name=="o" else "improved"}')

print('\ndates seen (should both be 2017-07-03 for Monday):')
print('original:', ts_o.dt.date.value_counts().head(3).to_dict())
print('improved:', ts_i.dt.date.value_counts().head(3).to_dict())

print('\nhour ranges: original', int(ts_o.dt.hour.min()), '-', int(ts_o.dt.hour.max()),
      '| improved', int(ts_i.dt.hour.min()), '-', int(ts_i.dt.hour.max()))

mo = set(ts_o.dt.floor('min').dropna())
mi_base = ts_i.dt.floor('min').dropna()
print(f'\ndistinct minutes: original {len(mo):,} | improved {len(set(mi_base)):,}')

print('\nminute-set overlap by hour shift applied to ORIGINAL:')
best, best_n = 0, -1
for shift in range(-12, 13):
    n = len({m + pd.Timedelta(hours=shift) for m in mo} & set(mi_base))
    flag = ''
    if n > best_n:
        best, best_n = shift, n
        flag = '  <-- best so far'
    print(f'  {shift:+3d}h  shared minutes {n:>5,}{flag}')
print(f'\nbest shift {best:+d}h sharing {best_n:,} minutes')

original  dayfirst=True   month==7 100.0%  NaT 0.00%
improved  dayfirst=False  month==7 100.0%  NaT 0.00%
12h repair applied to original

dates seen (should both be 2017-07-03 for Monday):
original: {datetime.date(2017, 7, 3): 529918}
improved: {datetime.date(2017, 7, 3): 371624}

hour ranges: original 8 - 17 | improved 11 - 20

distinct minutes: original 487 | improved 487

minute-set overlap by hour shift applied to ORIGINAL:
  -12h  shared minutes     0  <-- best so far
  -11h  shared minutes     0
  -10h  shared minutes     0
   -9h  shared minutes     0
   -8h  shared minutes     0
   -7h  shared minutes     0
   -6h  shared minutes     0
   -5h  shared minutes     7  <-- best so far
   -4h  shared minutes    67  <-- best so far
   -3h  shared minutes   127  <-- best so far
   -2h  shared minutes   187  <-- best so far
   -1h  shared minutes   247  <-- best so far
   +0h  shared minutes   307  <-- best so far
   +1h  shared minutes   367  <-- best so far
   +2h  shared minutes   4

In [5]:
# --- 4. Full-key match rate at the best clock shift --------------------------
KEY5 = ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol']

def keyframe(df, ts, shift_h=0):
    k = df[KEY5].copy()
    for c in ['src_ip', 'dst_ip']:
        k[c] = k[c].astype(str).str.strip()
    for c in ['src_port', 'dst_port', 'protocol']:
        k[c] = pd.to_numeric(k[c], errors='coerce').astype('Int64').astype(str)
    k['minute'] = (ts + pd.Timedelta(hours=shift_h)).dt.floor('min').astype(str)
    return k

iset = set(map(tuple, keyframe(im, ts_i).astype(str).values))
for shift in [0, best]:
    ko = keyframe(om, ts_o, shift)
    n = sum(1 for t in map(tuple, ko.astype(str).values) if t in iset)
    print(f'shift {shift:+3d}h  full-key matches {n:>9,}  ({n/len(om):.1%} of Monday)')

print()
print('READING THE RESULTS:')
print('- a component with ~0% shared values in section 2  -> encoding mismatch there')
print('- dates disagree in section 3                      -> parse problem')
print('- minute overlap ~0 at every shift                 -> clock beyond a constant offset')
print('- minutes align but full key ~0                    -> 5-tuple encoding problem')
print('Send this whole output back.')

shift  +0h  full-key matches     3,077  (0.6% of Monday)
shift  +3h  full-key matches   410,996  (77.6% of Monday)

READING THE RESULTS:
- a component with ~0% shared values in section 2  -> encoding mismatch there
- dates disagree in section 3                      -> parse problem
- minute overlap ~0 at every shift                 -> clock beyond a constant offset
- minutes align but full key ~0                    -> 5-tuple encoding problem
Send this whole output back.
